In [0]:
%run ./Includes/Copy-Datasets

In [0]:
from pyspark.sql import functions as F

def process_books_sales():
    orders_df = (spark.readStream.table("orders_silver")
                                .withColumn("book", F.explode(F.col("books"))))
    
    books_df = spark.read.table("current_books")

    query = (orders_df
                    .join(books_df, orders_df.book.book_id == books_df.book_id, "inner")
                    .writeStream
                        .outputMode("append")
                        .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/books_sales")
                        .trigger(availableNow=True)
                        .table("books_sales")
            )
    
    query.awaitTermination()

process_books_sales()


In [0]:
%sql
SELECT * FROM books_sales

In [0]:
%sql
SELECT COUNT(*) FROM books_sales

In [0]:
bookstore.load_new_data()
bookstore.process_bronze()
bookstore.process_books_silver()
bookstore.process_current_books()

process_books_sales()

In [0]:
%sql
SELECT count(*) FROM books_sales

In [0]:
bookstore.process_orders_silver()

process_books_sales()

In [0]:
%sql
SELECT count(*) FROM books_sales